In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
data=pd.read_csv("master_stock_data.csv")
len(data)

2219678

In [18]:
data.head()
data.isna().sum()

Ticker          0
Adj Close       0
Close           0
Dividends       0
High            0
Low             0
Open            0
Stock Splits    0
Volume          0
Day             0
Month           0
Year            0
quantity_1      0
dtype: int64

In [19]:
data.duplicated().sum()

np.int64(0)

In [23]:
data.head()

,Ticker,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume,Day,Month,Year,quantity_1
0,A,26.189770,31.473534,0.0,35.765381,28.612303,32.546494,0.0,62546380,18,11,1999,5
1,A,24.032093,28.880545,0.0,30.758226,28.478184,30.713518,0.0,15234146,19,11,1999,5
2,A,26.189770,31.473534,0.0,31.473534,28.657009,29.551144,0.0,6577870,22,11,1999,5
3,A,23.808886,28.612303,0.0,31.205294,28.612303,30.400572,0.0,5975611,23,11,1999,5
4,A,24.441307,29.372318,0.0,29.998213,28.612303,28.701717,0.0,4843231,24,11,1999,5


In [25]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
data["quantity_1"] = pd.cut(
    data["Volume"], bins=[0, 2, 4, 8, 10, np.inf], labels=[1, 2, 3, 4, 5]
)
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(data, data["quantity_1"]):
    strat_train_set = data.iloc[train_index]  
    strat_test_set = data.iloc[test_index]  


In [27]:
from sklearn.pipeline import Pipeline
mypipeline = Pipeline([ 
    ("impute", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler()),
])

In [59]:
data_no=strat_train_set.drop("Ticker",axis=1)
data_num_only = data_no.select_dtypes(include=["number"])
scaled_array = mypipeline.fit_transform(data_num_only)
scaled_data = pd.DataFrame(scaled_array, columns=data_num_only.columns, index=data_num_only.index)
scaled_data.head()

,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume,Day,Month,Year
1574667,-0.328864,-0.384225,-0.030834,-0.385267,-0.385663,-0.386458,-0.011875,-0.086796,-0.313820,-1.323182,0.736292
523257,-0.418685,-0.468839,-0.030834,-0.468958,-0.469447,-0.469503,-0.011875,-0.185396,-0.313820,1.599148,-0.677630
1769401,-0.382381,-0.424794,-0.030834,-0.424452,-0.426091,-0.425219,-0.011875,-0.180685,1.400876,-0.446483,-0.744959
1442602,-0.392877,-0.420425,-0.030834,-0.421073,-0.420712,-0.421331,-0.011875,-0.177370,-1.571264,1.306915,-1.552914
1406113,-0.421949,-0.459819,-0.030834,-0.459837,-0.459650,-0.459567,-0.011875,-0.173508,1.400876,0.137983,-1.956892


In [62]:
from sklearn.tree import DecisionTreeRegressor
X=scaled_data.drop("Adj Close",axis=1)
Y=scaled_data["Adj Close"]
X_encoded=pd.get_dummies(X)
model = DecisionTreeRegressor(random_state=40)
model.fit(X_encoded, Y)

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,40
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [63]:
# data test
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train, X_test, Y_train, Y_test = train_test_split(
    X_encoded,
    Y,
    test_size=0.2,
    random_state=40
)

model = DecisionTreeRegressor(random_state=40)

model.fit(X_train, Y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(Y_test, predictions)

mse = mean_squared_error(Y_test, predictions)

r2 = r2_score(Y_test, predictions)

print("Mean Absolute Error:", mae)
print("Mean Squared Error:", mse)
print("R2 Score:", r2)

Mean Absolute Error: 0.025165971591715915
Mean Squared Error: 0.0027500746507295288
R2 Score: 0.9971854769187614
